In [ ]:
# 필요한 패키지 설치 주석
# pip install fastapi uvicorn ollama google-generativeai python-multipart python-dotenv pillow mysql-connector-python nest_asyncio

import os
import io
import base64
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
import ollama
import google.generativeai as genai
from PIL import Image
import database

# .env 파일 로드
load_dotenv()

app = FastAPI()

# CORS 설정: 모든 Origin, Method, Header 허용
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def health_check():
    """서버 상태를 확인하는 엔드포인트입니다."""
    return {"status": "ok", "message": "AI 분석 서버가 가동 중입니다."}

@app.post("/analyze")
async def analyze_image(image: UploadFile = File(...), question: str = Form("이미지 내용을 분석해주세요.")):
    """이미지와 질문을 받아 분석 결과를 반환하는 API입니다."""
    try:
        # 이미지 데이터 읽기
        image_content = await image.read()
        
        # 환경 변수에서 모델 설정 로드
        use_model = os.getenv("USE_MODEL", "OLLAMA")
        response_text = ""
        
        if use_model == "OLLAMA":
            # Ollama llava 모델 사용
            model_name = os.getenv("OLLAMA_MODEL", "llava")
            ollama_response = ollama.chat(
                model=model_name,
                messages=[{
                    'role': 'user',
                    'content': question,
                    'images': [image_content]
                }]
            )
            response_text = ollama_response['message']['content']
            
        elif use_model == "GEMINI":
            # Google Gemini 2.0 Flash 사용
            api_key = os.getenv("GEMINI_API_KEY")
            if not api_key:
                return {"success": false, "message": "GEMINI_API_KEY가 설정되지 않았습니다."}
            
            genai.configure(api_key=api_key)
            model = genai.GenerativeModel('gemini-2.0-flash')
            
            # 이미지 변환
            img = Image.open(io.BytesIO(image_content))
            
            # 분석 요청
            gemini_response = model.generate_content([question, img])
            response_text = gemini_response.text
        
        else:
            return {"success": false, "message": "잘못된 USE_MODEL 설정입니다. (OLLAMA 또는 GEMINI)"}

        # 데이터베이스에 분석 결과 저장
        database.saveAiResponse(question, response_text, use_model)
        
        return {
            "success": true,
            "model": use_model,
            "result": response_text
        }
        
    except Exception as e:
        return {"success": false, "message": f"분석 중 에러가 발생했습니다: {str(e)}"}


In [ ]:
import nest_asyncio
import uvicorn

# Jupyter Notebook 환경 호환성을 위한 설정
nest_asyncio.apply()

if __name__ == "__main__":
    # 서버 실행
    uvicorn.run(app, host="0.0.0.0", port=8000)
